# 01 - Data Discovery

In [ ]:
import os
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

raw = Path('../../Data/Raw')
print(raw.exists())

In [ ]:
# list all images/labels
images = []
labels = []
for p in raw.rglob('*'):
    if p.is_file():
        ext = p.suffix.lower()
        if ext in ['.jpg','.jpeg','.png','.bmp']:
            images.append(p)
        elif ext == '.txt':
            labels.append(p)
print('images:', len(images))
print('labels:', len(labels))

In [ ]:
# count by class
from collections import Counter
c = Counter()
for im in images:
    # class is parent of images folder
    c[im.parent.parent.name] += 1
c

In [ ]:
# look at a label
lab = labels[0]
print(lab)
with open(lab) as f:
    print(f.read())

In [ ]:
# show 9 images with boxes
import random
random.seed(42)
sample = random.sample(images, min(9, len(images)))
plt.figure(figsize=(12,12))
for i,im in enumerate(sample):
    img = cv2.imread(str(im))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h,w = img.shape[:2]
    labp = im.parent.parent / 'labels' / (im.stem + '.txt')
    if labp.exists():
        with open(labp) as f:
            for line in f:
                line = line.strip()
                if not line: continue
                parts = line.split()
                cls = parts[0]
                x,y,bw,bh = map(float, parts[1:5])
                x1 = int((x-bw/2)*w); y1 = int((y-bh/2)*h)
                x2 = int((x+bw/2)*w); y2 = int((y+bh/2)*h)
                cv2.rectangle(img,(x1,y1),(x2,y2),(255,0,0),2)
                cv2.putText(img, cls,(x1,max(0,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,0,0),1)
    plt.subplot(3,3,i+1)
    plt.imshow(img)
    plt.axis('off')
    plt.title(im.parent.parent.name)
plt.tight_layout()
plt.show()

In [ ]:
# class distribution
import pandas as pd
df = pd.DataFrame({'path': [str(x) for x in images], 'cls': [x.parent.parent.name for x in images]})
df['cls'].value_counts()